# Interrupts 中断
当触发中断时，LangGraph 会使用其持久层保存图的状态，并无限期地等待恢复执行。
在图的任意节点调用 `interrupt()`  函数。该函数接受任何可序列化的 JSON 值，并将其传递给调用者。准备继续执行时，可以使用 `Command` 重新调用图来恢复执行，此时该值将成为节点内部 `interrupt() ` 调用的返回值。
- 检查点机制会保留你的进度： 检查点会写入精确的图状态，以便你即使在错误状态下也能稍后恢复。
- `thread_id` 是你的指针： 设置 `config={"configurable": {"thread_id": ...}}` 以告诉检查指针要加载哪个状态。
- 中断有效负载通过 `chunk["interrupts"]` 浮现： 当使用 `version="v2"` 进行流式传输时，传递给 `interrupt()` 值会出现在 `values` 流部分的 `interrupts` 字段中，以便知道图表正在等待什么。


## Pause using interrupt 使用 interrupt 暂停
1. 用于持久化图状态的检查点 （在生产环境中使用持久化检查点）
2. 配置中需要包含线程 ID ，以便运行时知道要从哪个状态恢复。
3. 要在需要暂停的地方调用 interrupt() （有效载荷必须是可序列化的 JSON）

In [ ]:
from langgraph.types import interrupt

class State:
    pass

def approval_node(state: State):
    # Pause and ask for approval
    approved = interrupt("Do you approve this action?")

    # When you resume, Command(resume=...) returns that value here
    return {"approved": approved}

调用 interrupt 时，会发生以下情况：
1. 图执行会在调用 `interrupt` 确切位置暂停。
2. 状态通过检查点保存 ，以便稍后恢复执行。在生产环境中，这应该是一个持久化的检查点（例如，由数据库支持）。
3. 在 `__interrupt__ `事件下， 该值会返回给调用者；它可以是任何可序列化为 JSON 的值（字符串、对象、数组等）。
4. 图表会无限期地等待， 直到您通过响应恢复执行。
5. 恢复运行时， 响应会传递回节点，成为 `interrupt()` 调用的返回值。

## Resuming interrupts  恢复中断
可以通过再次调用该图并传入包含恢复值的 Command 来恢复执行。恢复值会传递回 interrupt 调用，从而允许节点使用外部输入继续执行。


In [ ]:
from langgraph.types import Command

# Initial run - hits the interrupt and pauses
# thread_id is the persistent pointer (stores a stable ID in production)
config = {"configurable": {"thread_id": "thread-1"}}
result = graph.invoke({"input": "data"}, config=config, version="v2") # type: ignore

# result is a GraphOutput with .value and .interrupts
# .interrupts contains the payloads passed to interrupt()
print(result.interrupts)
# > (Interrupt(value='Do you approve this action?'),)

# Resume with the human's response
# The resume payload becomes the return value of interrupt() inside the node
graph.invoke(Command(resume=True), config=config, version="v2")# type: ignore

- 恢复运行时，必须使用与中断发生时相同的线程 ID。
- 传递给 `Command(resume=...)` 值将成为 `interrupt` 调用的返回值。
- 节点恢复运行时，会从 `interrupt` 发生时的节点起始位置重新开始执行，因此 `interrupt`之前的任何代码都会再次运行。
- 您可以将任何可序列化的 JSON 值作为恢复值传递。

## Common patterns  常见模式
### Stream with human-in-the-loop (HITL) interrupts 带人机交互中断（HITL）的流
在构建具有人机交互工作流程的交互式代理时，您可以同时传输消息块和节点更新，以便在处理中断的同时提供实时反馈。
- 实时流式传输生成的 AI 响应
- 检测图表何时遇到中断
- 无缝处理用户输入并恢复执行

In [ ]:
async for chunk in graph.astream(# type: ignore
    initial_input,# type: ignore
    stream_mode=["messages", "updates"],
    subgraphs=True,
    config=config,
    version="v2",
):
    if chunk["type"] == "messages":
        # Handle streaming message content
        msg, _ = chunk["data"]
        if isinstance(msg, AIMessageChunk) and msg.content:# type: ignore
            display_streaming_content(msg.content)# type: ignore

    elif chunk["type"] == "updates":
        # Check for interrupts in the updates data
        if "__interrupt__" in chunk["data"]:
            interrupt_info = chunk["data"]["__interrupt__"][0].value
            user_response = get_user_input(interrupt_info)# type: ignore
            initial_input = Command(resume=user_response)
            break
        else:
            current_node = list(chunk["data"].keys())[0]

- `version="v2"` ：所有数据块都是 StreamPart 字典，包含 type 、 ns 和 data 键。
- `chunk["type"]` ：根据流模式（ "messages" 、 "updates" 等）进行类型推断。
- `chunk["ns"]` ：标识源图（空元组表示根图，填充表示子图）
- `subgraphs=True` ：嵌套图中的中断检测必需
- `Command(resume=...)` ：使用用户提供的数据恢复图的执行

### Handling multiple interrupts 处理多个中断
当并行分支同时发生中断时（例如，扇出到多个节点，每个节点都 `interrupt()` 函数），可能需要在一次调用中恢复多个中断。在一次调用中恢复多个中断时，请将每个中断 ID 映射到其恢复值。这样可以确保每个响应在运行时都与正确的中断配对。

In [ ]:
from typing import Annotated, TypedDict
import operator

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import START, END, StateGraph
from langgraph.types import Command, interrupt


class State(TypedDict):
    vals: Annotated[list[str], operator.add]


def node_a(state):
    answer = interrupt("question_a")
    return {"vals": [f"a:{answer}"]}


def node_b(state):
    answer = interrupt("question_b")
    return {"vals": [f"b:{answer}"]}


graph = (
    StateGraph(State)
    .add_node("a", node_a)
    .add_node("b", node_b)
    .add_edge(START, "a")
    .add_edge(START, "b")
    .add_edge("a", END)
    .add_edge("b", END)
    .compile(checkpointer=InMemorySaver())
)

config = {"configurable": {"thread_id": "1"}}

# Step 1: invoke - both parallel nodes hit interrupt() and pause
interrupted_result = graph.invoke({"vals": []}, config)
print(interrupted_result)
"""
{
    'vals': [],
    '__interrupt__': [
        Interrupt(value='question_a', id='bd4f3183600f2c41dddafbf8f0f7be7b'),
        Interrupt(value='question_b', id='29963e3d3585f0cef025dd0f14323f55')
    ]
}
"""

# Step 2: resume all pending interrupts at once
resume_map = {
    i.id: f"answer for {i.value}"
    for i in interrupted_result["__interrupt__"]
}
result = graph.invoke(Command(resume=resume_map), config)

print("Final state:", result)
#> Final state: {'vals': ['a:answer for question_a', 'b:answer for question_b']}

### Approve or reject  批准或拒绝
在执行关键操作前暂停并请求批准。例如，可能需要请人批准 API 调用、数据库更改或任何其他重要决策。

In [ ]:
from typing import Literal
from langgraph.types import interrupt, Command

def approval_node(state: State) -> Command[Literal["proceed", "cancel"]]:
    # Pause execution; payload shows up under result["__interrupt__"]
    is_approved = interrupt({
        "question": "Do you want to proceed with this action?",
        "details": state["action_details"]
    })

    # Route based on the response
    if is_approved:
        return Command(goto="proceed")  # Runs after the resume payload is provided
    else:
        return Command(goto="cancel")

### Review and edit state  审查和编辑状态
有时,需要先让人工审核和编辑部分图表状态，然后再继续。这对于修正 LLM、添加缺失信息或进行调整非常有用。

In [ ]:
from langgraph.types import interrupt

def review_node(state: State):
    # Pause and show the current content for review (surfaces in result["__interrupt__"])
    edited_content = interrupt({
        "instruction": "Review and edit this content",
        "content": state["generated_text"]
    })

    # Update the state with the edited version
    return {"generated_text": edited_content}

### Interrupts in tools  工具中断
可以直接在工具函数内部放置中断。这样，每次调用工具时，工具本身都会暂停以等待批准，从而允许在执行工具调用之前进行人工审核和编辑。

In [ ]:
from langchain.tools import tool
from langgraph.types import interrupt

@tool
def send_email(to: str, subject: str, body: str):
    """Send an email to a recipient."""

    # Pause before sending; payload surfaces in result["__interrupt__"]
    response = interrupt({
        "action": "send_email",
        "to": to,
        "subject": subject,
        "body": body,
        "message": "Approve sending this email?"
    })

    if response.get("action") == "approve":
        # Resume value can override inputs before executing
        final_to = response.get("to", to)
        final_subject = response.get("subject", subject)
        final_body = response.get("body", body)
        return f"Email sent to {final_to} with subject '{final_subject}'"
    return "Email cancelled by user"

### Validating human input  验证人工输入
有时需要验证用户输入的内容，如果无效则需要再次询问。可以使用循环中的多次 `interrupt` 调用来实现这一点。

In [ ]:
from langgraph.types import interrupt

def get_age_node(state: State):
    prompt = "What is your age?"

    while True:
        answer = interrupt(prompt)  # payload surfaces in result["__interrupt__"]

        # Validate the input
        if isinstance(answer, int) and answer > 0:
            # Valid input - continue
            break
        else:
            # Invalid input - ask again with a more specific prompt
            prompt = f"'{answer}' is not a valid age. Please enter a positive number."

    return {"age": answer}

## Rules of interrupts  中断规则
在节点内调用 `interrupt` 时，LangGraph 会抛出一个异常来暂停执行，该异常会通知运行时暂停。此异常会沿着调用栈向上传播，并被运行时捕获，运行时会通知图保存当前状态并等待外部输入。
当程序恢复执行（在您提供所需输入后）时，运行时会从头开始重新启动整个节点，而不是从调用 `interrupt` 的确切代码行恢复执行。这意味着 `interrupt` 之前运行的任何代码都会再次执行。因此，在使用中断时需要遵循一些重要的规则，以确保其行为符合预期。
#### Do not wrap interrupt calls in try/except 不要将 interrupt 包装在 try/except 语句中。
`interrupt` 之所以会在调用时暂停执行，是通过抛出一个特殊异常来实现的。如果将 `interrupt` 调用放在 try/except 代码块中，就可以捕获到这个异常，并且不会将中断传递回程序图。
- 将 interrupt 调用与容易出错的代码分开
- 在 try/except 代码块中使用特定的异常类型

In [ ]:
def node_a(state: State):
    # ✅ Good: interrupting first, then handling
    # error conditions separately
    interrupt("What's your name?")
    try:
        fetch_data()  # type: ignore
    except Exception as e:
        print(e)
    return state

In [ ]:
def node_a(state: State):
    # ✅ Good: catching specific exception types
    # will not catch the interrupt exception
    try:
        name = interrupt("What's your name?")
        fetch_data()  # type: ignore # This can fail
    except NetworkException as e: # type: ignore
        print(e)
    return state

#### Do not reorder interrupt calls within a node 不要重新排列节点内的 interrupt 调用顺序
当一个节点包含多个中断调用时，LangGraph 会维护一个与执行该节点的任务相关的恢复值列表。每次执行恢复时，都会从节点的开头开始。对于遇到的每个中断，LangGraph 都会检查任务的恢复列表中是否存在匹配的值。匹配严格基于索引 ，因此节点内中断调用的顺序至关重要。
- 保持所有节点执行过程中 `interrupt` 调用的一致性

In [ ]:
def node_a(state: State):
    # ✅ Good: interrupt calls happen in the same order every time
    name = interrupt("What's your name?")
    age = interrupt("What's your age?")
    city = interrupt("What's your city?")

    return {
        "name": name,
        "age": age,
        "city": city
    }

- 不要有条件地跳过节点内的 `interrupt` 调用
- 不要使用在每次执行中都不确定的逻辑来循环 `interrupt`

In [ ]:
def node_a(state: State):
    # ❌ Bad: conditionally skipping interrupts changes the order
    name = interrupt("What's your name?")

    # On first run, this might skip the interrupt
    # On resume, it might not skip it - causing index mismatch
    if state.get("needs_age"):
        age = interrupt("What's your age?")

    city = interrupt("What's your city?")

    return {"name": name, "city": city}

In [ ]:
def node_a(state: State):
    # ❌ Bad: looping based on non-deterministic data
    # The number of interrupts changes between executions
    results = []
    for item in state.get("dynamic_list", []):  # List might change between runs
        result = interrupt(f"Approve {item}?")
        results.append(result)

    return {"results": results}

#### Do not return complex values in interrupt calls 不要在 interrupt 调用中返回复杂值
根据所使用的检查点类型，复杂值可能无法序列化（例如，函数无法序列化）。为了使图表能够适应任何部署环境，最佳实践是仅使用可以合理序列化的值。
- 将简单的、可序列化的 JSON 类型传递给 `interrupt`
- 传递包含简单值的字典/对象

In [ ]:
def node_a(state: State):
    # ✅ Good: passing simple types that are serializable
    name = interrupt("What's your name?")
    count = interrupt(42)
    approved = interrupt(True)

    return {"name": name, "count": count, "approved": approved}

In [ ]:
def node_a(state: State):
    # ✅ Good: passing dictionaries with simple values
    response = interrupt({
        "question": "Enter user details",
        "fields": ["name", "email", "age"],
        "current_values": state.get("user", {})
    })

    return {"user": response}

- 请勿将函数、类实例或其他复杂对象传递给 `interrupt`

In [ ]:
def validate_input(value):
    return len(value) > 0

def node_a(state: State):
    # ❌ Bad: passing a function to interrupt
    # The function cannot be serialized
    response = interrupt({
        "question": "What's your name?",
        "validator": validate_input  # This will fail
    })
    return {"name": response}

In [ ]:
class DataProcessor:
    def __init__(self, config):
        self.config = config

def node_a(state: State):
    processor = DataProcessor({"mode": "strict"})

    # ❌ Bad: passing a class instance to interrupt
    # The instance cannot be serialized
    response = interrupt({
        "question": "Enter data to process",
        "processor": processor  # This will fail
    })
    return {"result": response}

#### Side effects called before interrupt must be idempotent 在 interrupt 之前调用的副作用必须是幂等的
由于中断的工作原理是重新运行调用它的节点，因此 `interrupt` 之前调用的副作用（理想情况下）应该是幂等的。幂等性是指同一个操作可以多次执行，而结果不会改变。
例如，可能需要调用 API 来更新节点内的一条记录。如果在调用该 API 后 `interrupt` ，则当节点恢复运行时，该 API 调用将被多次重新执行，这可能会覆盖初始更新或创建重复记录。
- 在 `interrupt` 之前使用幂等操作
- `interrupt` 通话后放置副作用
- 尽可能将副作用分离到单独的节点中

In [ ]:
def node_a(state: State):
    # ✅ Good: using upsert operation which is idempotent
    # Running this multiple times will have the same result
    db.upsert_user( # type: ignore
        user_id=state["user_id"],
        status="pending_approval"
    )

    approved = interrupt("Approve this change?")

    return {"approved": approved}

In [ ]:
def node_a(state: State):
    # ✅ Good: placing side effect after the interrupt
    # This ensures it only runs once after approval is received
    approved = interrupt("Approve this change?")

    if approved:
        db.create_audit_log(# type: ignore
            user_id=state["user_id"],
            action="approved"
        )

    return {"approved": approved}

In [ ]:
def approval_node(state: State):
    # ✅ Good: only handling the interrupt in this node
    approved = interrupt("Approve this change?")

    return {"approved": approved}

def notification_node(state: State):
    # ✅ Good: side effect happens in a separate node
    # This runs after approval, so it only executes once
    if (state.approved):
        send_notification(# type: ignore
            user_id=state["user_id"],
            status="approved"
        )

    return state

##  Using with subgraphs called as functions 使用称为函数的子图
当在节点内调用子图时，父图将从调用子图并触发 interrupt 节点的开头恢复执行。同样， 子图也将从调用 interrupt 节点的开头恢复执行。

In [ ]:
def node_in_parent_graph(state: State):
    some_code()  # type: ignore # <-- This will re-execute when resumed
    # Invoke a subgraph as a function.
    # The subgraph contains an `interrupt` call.
    subgraph_result = subgraph.invoke(some_input) # type: ignore
    # ...

def node_in_subgraph(state: State):
    some_other_code()  # type: ignore # <-- This will also re-execute when resumed
    result = interrupt("What's your name?")
    # ...

## Debugging with interrupts 使用中断进行调试
要调试和测试图，可以使用静态中断作为断点，逐个节点地执行图的执行过程。静态中断会在节点执行之前或之后的特定点触发。您可以在编译图时指定 `interrupt_before` 和 `interrupt_after` 来设置这些点。
> 对于人机交互工作流程， 不建议使用静态中断。请改用 `interrupt` 函数。

In [ ]:
graph = builder.compile( # type: ignore
    interrupt_before=["node_a"],
    interrupt_after=["node_b", "node_c"],
    checkpointer=checkpointer, # type: ignore
)

# Pass a thread ID to the graph
config = {
    "configurable": {
        "thread_id": "some_thread"
    }
}

# Run the graph until the breakpoint
graph.invoke(inputs, config=config) # type: ignore

# Resume the graph
graph.invoke(None, config=config)

1. 断点是在 `compile` 时设置的。
2. `interrupt_before` 指定在执行节点之前应该暂停执行的节点。
3. `interrupt_after `指定执行在节点执行完毕后应暂停的节点。
4. 需要检查点才能启用断点。
5. 该图表会一直运行，直到遇到第一个断点为止。 
6. 将输入值设为 None 即可恢复图表运行。这将使图表一直运行到遇到下一个断点为止。

In [ ]:
config = {
    "configurable": {
        "thread_id": "some_thread"
    }
}

# Run the graph until the breakpoint
graph.invoke(
    inputs,# type: ignore
    interrupt_before=["node_a"],
    interrupt_after=["node_b", "node_c"],
    config=config,
)

# Resume the graph
graph.invoke(None, config=config)

1. `graph.invoke` 调用时会传入 `interrupt_before` 和 `interrupt_after` 参数。这是一个运行时配置，每次调用时都可以更改。
2. `interrupt_before` 指定在执行节点之前应该暂停执行的节点。
3. `interrupt_after` 指定执行在节点执行完毕后应暂停的节点。
4. 该图表会一直运行，直到遇到第一个断点为止。
5. 将输入值设为 None 即可恢复图表运行。这将使图表一直运行到遇到下一个断点为止。